# 06 — Step 5: Causal Validation

**Goal**: Zero-ablate each candidate reflection head and measure if reflection quality
degrades while overall image quality is preserved.

If ablating a head specifically degrades the reflection region but not the object region,
that head is causally involved in generating reflections.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/iker0/mi-mirror/blob/main/notebook_demo/06_step5_causal_validation.ipynb
)

In [ ]:
# Clone repo and set up paths (run once on Colab)
!git clone https://github.com/iker0/mi-mirror.git /content/mi-mirror 2>/dev/null || echo "Already cloned"
%cd /content/mi-mirror

In [ ]:
import sys
sys.path.insert(0, '..')

import pickle
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from collections import OrderedDict

from scripts.config import (
    EXPERIMENTS_DIR, MIRROR_PROMPTS, SEEDS, MIRROR_DIR, PROMPT_OBJECTS,
    NUM_DUAL_STREAM_BLOCKS,
)
from scripts.model_loader import load_flux_pipeline
from scripts.roi import get_object_and_reflection_rois
from scripts.generate import generate_with_ablation, generate_baseline
from scripts.metrics import compute_reflection_quality
from scripts.visualization import plot_ablation_comparison, plot_degradation_bars

STEP1_DIR = EXPERIMENTS_DIR / "step1_selectivity"
STEP3_DIR = EXPERIMENTS_DIR / "step3_temporal"
STEP5_DIR = EXPERIMENTS_DIR / "step5_causal"
STEP5_DIR.mkdir(parents=True, exist_ok=True)

# Load previous results
with open(STEP1_DIR / "step1_results.pkl", "rb") as f:
    step1 = pickle.load(f)
candidates = step1["candidates"]

with open(STEP3_DIR / "step3_results.pkl", "rb") as f:
    step3 = pickle.load(f)
peak_timestep = step3["peak_timestep"]

# Top-5 candidates for ablation
top5 = candidates[:5]
print(f"Top-5 candidates: {[(b, h, f'{s:.4f}') for b, h, s in top5]}")
print(f"Peak timestep: {peak_timestep}")

In [ ]:
pipe = load_flux_pipeline(quantize_nf4=True, cpu_offload=True)
print("Model loaded.")

In [ ]:
# Use 3 prompt/seed pairs for statistical robustness
test_configs = [
    (0, SEEDS[0]),  # prompt 0, seed 42
    (1, SEEDS[1]),  # prompt 1, seed 137
    (2, SEEDS[2]),  # prompt 2, seed 256
]

# Generate baseline images and get ROIs
baselines = {}
rois = {}
for prompt_idx, seed in test_configs:
    prompt = MIRROR_PROMPTS[prompt_idx]
    obj_name = PROMPT_OBJECTS[prompt_idx]
    
    # Load pre-generated image for ROI extraction
    img_path = MIRROR_DIR / f"prompt{prompt_idx:02d}_seed{seed}.png"
    if img_path.exists():
        ref_image = Image.open(img_path)
    else:
        ref_image = generate_baseline(pipe, prompt, seed)
    
    obj_roi, ref_roi = get_object_and_reflection_rois(ref_image, obj_name)
    
    # Generate baseline with same seed (deterministic)
    baseline = generate_baseline(pipe, prompt, seed)
    baselines[(prompt_idx, seed)] = baseline
    rois[(prompt_idx, seed)] = (obj_roi, ref_roi)
    
    print(f"Baseline {prompt_idx}/{seed}: obj={obj_roi.size} tokens, ref={ref_roi.size} tokens")

print(f"\nGenerated {len(baselines)} baselines.")

In [ ]:
# Single-head ablation loop
# For each top-5 candidate, ablate it and measure degradation

all_metrics = {}  # (block, head) -> list of metrics dicts (one per test config)
all_ablated_images = {}  # (block, head, prompt_idx, seed) -> image

for b, h, s in top5:
    print(f"\n--- Ablating B{b}H{h} (selectivity={s:.4f}) ---")
    head_metrics = []
    
    for prompt_idx, seed in test_configs:
        prompt = MIRROR_PROMPTS[prompt_idx]
        ablation_targets = [(b, [h])]
        
        ablated = generate_with_ablation(
            pipe, prompt, seed, ablation_targets=ablation_targets
        )
        
        obj_roi, ref_roi = rois[(prompt_idx, seed)]
        metrics = compute_reflection_quality(
            baselines[(prompt_idx, seed)], ablated, ref_roi, obj_roi
        )
        head_metrics.append(metrics)
        all_ablated_images[(b, h, prompt_idx, seed)] = ablated
        
        print(f"  P{prompt_idx}/S{seed}: MSE_ref={metrics['mse_reflection']:.6f}, "
              f"MSE_obj={metrics['mse_object']:.6f}, "
              f"deg_ratio={metrics['degradation_ratio']:.2f}")
    
    all_metrics[(b, h)] = head_metrics

print("\nAblation complete.")

In [ ]:
# Visualize: side-by-side original vs ablated (first test config)
prompt_idx, seed = test_configs[0]
original = baselines[(prompt_idx, seed)]

ablated_images_display = OrderedDict()
for b, h, s in top5:
    ablated_images_display[(b, h)] = all_ablated_images[(b, h, prompt_idx, seed)]

fig = plot_ablation_comparison(
    original, ablated_images_display,
    title=f"Single-Head Ablation (Prompt {prompt_idx}, Seed {seed})",
)
fig.savefig(STEP5_DIR / "ablation_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Aggregate metrics across test configs and plot degradation bars
import numpy as np

avg_metrics = OrderedDict()
for (b, h), metrics_list in all_metrics.items():
    avg = {}
    for key in metrics_list[0]:
        avg[key] = float(np.mean([m[key] for m in metrics_list]))
    avg_metrics[(b, h)] = avg

print("Averaged metrics across test configs:")
print(f"{'Head':>8} {'MSE_ref':>10} {'MSE_obj':>10} {'SSIM_ref':>10} {'SSIM_obj':>10} {'Deg.Ratio':>10}")
for (b, h), m in avg_metrics.items():
    print(f"B{b}H{h:>2}   {m['mse_reflection']:>10.6f} {m['mse_object']:>10.6f} "
          f"{m['ssim_reflection']:>10.4f} {m['ssim_object']:>10.4f} {m['degradation_ratio']:>10.2f}")

fig = plot_degradation_bars(avg_metrics, title="Averaged Reflection vs Object Degradation")
fig.savefig(STEP5_DIR / "degradation_bars.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Rank confirmed reflection heads by degradation ratio
# Higher ratio = ablation specifically hurts reflection more than object

ranked = sorted(avg_metrics.items(), key=lambda x: x[1]["degradation_ratio"], reverse=True)

print("\n=== Confirmed Reflection Heads (ranked by degradation ratio) ===")
confirmed_heads = []
for (b, h), m in ranked:
    ratio = m["degradation_ratio"]
    status = "CONFIRMED" if ratio > 1.0 else "weak"
    print(f"  B{b}H{h}: degradation_ratio={ratio:.2f} [{status}]")
    if ratio > 1.0:
        confirmed_heads.append((b, h, ratio))

print(f"\n{len(confirmed_heads)} heads confirmed as reflection-specific.")

In [ ]:
# Save results
results = {
    "candidates": [(b, h, s) for b, h, s in top5],
    "test_configs": test_configs,
    "all_metrics": {f"B{b}H{h}": metrics for (b, h), metrics in all_metrics.items()},
    "avg_metrics": {f"B{b}H{h}": m for (b, h), m in avg_metrics.items()},
    "confirmed_heads": confirmed_heads,
}
with open(STEP5_DIR / "step5_results.pkl", "wb") as f:
    pickle.dump(results, f)

print(f"Results saved to {STEP5_DIR}")
print("\n=== Step 5 Complete ===")
print("Next: Step 6 — Circuit Composition Analysis")